### Machine Learning Fundamentals, XGBoost

1. Core Intuition and Boosting Assembly: XGBoost builds an ensemble step-by-step: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$. It starts with a base guess $F_0$ (like average house price or base log-odds $0.5$). Each new decision tree $h_m(x)$ predicts remaining errors (residuals) of the previous step. The learning rate $\eta$ scales down each tree's contribution to prevent overshooting.

2. Simple Residuals ($g_i$) and Weights ($h_i$): At each step, every data point $i$ calculates two values based on its current error:
* **Gradient ($g_i$):** Direction of error ($p_i - y_i$ for classification, $\hat{y}_i - y_i$ for regression). Negative gradient $-g_i$ is literally the residual error.
* **Hessian ($h_i$):** Confidence weight ($1$ for regression, $p_i(1 - p_i)$ for classification).

3. The Only Two Core Formulas: Instead of standard Gini or MSE, XGBoost uses $G = \sum g_i$ (total error in a node) and $H = \sum h_i$ (total confidence in a node):
* **Leaf Output Value ($w^*$):** $w^* = -\frac{G}{H + \lambda}$ (where $\lambda$ is L2 regularization that shrinks large outputs).
* **Node Similarity Score ($SS$):** $SS = \frac{G^2}{2(H + \lambda)}$ (measures how well the node isolates remaining error).

4. Evaluating Splits with Gain: To test if a feature split is worth making, XGBoost compares child node scores to the parent score:

$$\text{Gain} = \frac{1}{2}\left[ SS_L + SS_R - SS_{\text{Parent}} \right] - \gamma$$

If $\text{Gain} > 0$, the split is accepted. The hyperparameter $\gamma$ sets the minimum threshold to prevent useless splits.

5. Node Cover & Min Child Weight: Cover is simply the sum of Hessians in a node ($\text{Cover} = H = \sum h_i$). The parameter `min_child_weight` stops a split if either child ends up with $H < \text{min\_child\_weight}$. In regression, Cover is just row count; in classification, it is sample size weighted by uncertainty $p(1-p)$.

7. Bottom-Up Tree Pruning: Trees grow top-down to `max_depth`. Then XGBoost checks branches bottom-up: if $\text{Gain} < 0$ at a node, the split is pruned, and child leaves collapse back into a single leaf with weight $w^* = -\frac{G_{\text{Parent}}}{H_{\text{Parent}} + \lambda}$.

8. Missing Value Routing (Sparsity Awareness): If data points have missing values ($N/A$) for a feature, XGBoost tests sending all missing points to the Left branch, then to the Right branch. The path that produces the higher Gain is saved as the node's permanent default path for missing values.

9. Ensemble Output & Class Imbalance: Final predictions sum all log-odds $z = z_0 + \eta \sum h_m(x)$, converted to probability via $p = \frac{1}{1 + e^{-z}}$. Parameter `scale_pos_weight` ($w_{\text{pos}}$) multiplies $g_i$ and $h_i$ for positive samples ($y_i=1$), forcing the trees to prioritize minority class errors.

---


### Example: 
Classification Worked Example ($N=4$, $x=[1,2,8,9]$, $y=[0,0,1,1]$, $\lambda=0, \eta=0.3, \gamma=0$, initial log-odds $z_0=0 \implies p_0=0.5$):
* **Round 1:**
* $g_i = p_0 - y_i \implies g = [0.5, 0.5, -0.5, -0.5]$; $h_i = 0.5(1-0.5) = 0.25 \implies h = [0.25, 0.25, 0.25, 0.25]$.
* Split $x \le 5 \implies$ Left ($x \le 2$): $G_L = 1.0, H_L = 0.5$; Right ($x \ge 8$): $G_R = -1.0, H_R = 0.5$.
* Leaf weights: $w_L^* = -\frac{1.0}{0.5} = -2.0$; $w_R^* = -\frac{-1.0}{0.5} = +2.0$.
* $\text{Gain} = \frac{1}{2}\left[\frac{1^2}{0.5} + \frac{(-1)^2}{0.5} - 0\right] = 2.0 > 0$.
* Updates: $z_1 = z_0 + 0.3(w^*) \implies z_L = -0.6, z_R = +0.6 \implies p = [0.354, 0.354, 0.646, 0.646]$.

* **Round 2:**
* $g = [0.354, 0.354, -0.354, -0.354]$; $h = 0.354(0.646) = 0.229 \implies h = [0.229, 0.229, 0.229, 0.229]$.
* Split $x \le 5 \implies G_L = 0.708, H_L = 0.458 \implies w_L^* = -\frac{0.708}{0.458} = -1.546$; $w_R^* = +1.546$.
* $\text{Gain} = \frac{1}{2}\left[\frac{0.708^2}{0.458} + \frac{(-0.708)^2}{0.458} - 0\right] = 1.094 > 0$.
* Updates: $z_2 = z_1 + 0.3(w^*) \implies z_L = -1.064, z_R = +1.064 \implies p = [0.257, 0.257, 0.743, 0.743]$.

In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradient(y, p):    # g = p - y
    return p - y

def hessian(p):         # h = p(1-p)
    return p * (1 - p)

print("g:", gradient(np.array([1.0]), np.array([0.8])), "h:", hessian(np.array([0.8])))  # -0.2, 0.16

from sklearn.metrics import log_loss as sk_log_loss
y, z = np.array([1, 0, 1, 1, 0]), np.array([0.5, -1.2, 2.0, 0.1, -0.3])
p = sigmoid(z)
print("log loss (ours vs sklearn):", log_loss(y, p), sk_log_loss(y, p))

g: [-0.2] h: [0.16]
log loss (ours vs sklearn): 0.4126078734206417 0.4126078734206417


In [2]:
def leaf_score(g, h, lam=0.0):    # G^2 / (H + lambda)
    return (np.sum(g) ** 2) / (np.sum(h) + lam)

def leaf_value(g, h, lam=0.0):    # -G / (H + lambda)
    return -np.sum(g) / (np.sum(h) + lam)

print("leaf_score:", leaf_score(np.array([-0.2, -0.3]), np.array([0.16, 0.21])))  # ~0.676

leaf_score: 0.6756756756756757


In [ ]:
def best_split_xgb(X, g, h, lam=0.0):
    best_gain = -float("inf")
    best_feature, best_threshold = None, None
    parent_score = leaf_score(g, h, lam)

    for feature_idx in range(X.shape[1]):
        values = np.unique(X[:, feature_idx])
        thresholds = (values[:-1] + values[1:]) / 2

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            g_left, h_left = g[left_mask], h[left_mask]
            g_right, h_right = g[~left_mask], h[~left_mask]

            if len(g_left) == 0 or len(g_right) == 0:
                continue

            gain = leaf_score(g_left, h_left, lam) + leaf_score(g_right, h_right, lam) - parent_score

            if gain > best_gain:
                best_gain = gain
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gain

In [ ]:
X = np.array([1, 2, 8, 9], dtype=float).reshape(-1, 1)
y_toy = np.array([0, 0, 1, 1], dtype=float)

def boosting_round(X, y, z, lam=0.0, lr=0.3):
    p = sigmoid(z)
    g, h = gradient(y, p), hessian(p)
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    left_mask = X[:, feature] <= threshold
    left_value = leaf_value(g[left_mask], h[left_mask], lam)
    right_value = leaf_value(g[~left_mask], h[~left_mask], lam)
    tree_output = np.where(left_mask, left_value, right_value)
    return z + lr * tree_output, gain, left_value, right_value

z = np.zeros(4)
for round_num in range(1, 3):
    z, gain, left_val, right_val = boosting_round(X, y_toy, z)
    print(f"round {round_num}: gain={gain:.3f}, values=({left_val:.3f}, {right_val:.3f}), p={np.round(sigmoid(z), 3)}")
# expect: round1 gain=4.0, p=[.354,.354,.646,.646]; round2 gain~2.19, p=[.257,.257,.743,.743]

In [ ]:
from xgboost import XGBClassifier

# approximate check against the real library -- internals differ in minor ways, but
# direction and scale should agree with the worked example above
xgb_toy = XGBClassifier(n_estimators=2, max_depth=1, learning_rate=0.3, reg_lambda=0,
                          base_score=0.5, eval_metric="logloss")
xgb_toy.fit(X, y_toy)
print("xgboost p:", np.round(xgb_toy.predict_proba(X)[:, 1], 3))

In [ ]:
class TreeNode:
    def __init__(self, g, h, lam):
        self.value = leaf_value(g, h, lam)   # computed up front so pruning can collapse back to this
        self.is_leaf = True
        self.feature = self.threshold = self.gain = None
        self.left = self.right = None

def grow_tree(X, g, h, lam=1.0, max_depth=2, depth=0):
    node = TreeNode(g, h, lam)
    if depth >= max_depth or len(g) < 2:
        return node   # only stopping rule during growth is depth/size -- NOT gain
    feature, threshold, gain = best_split_xgb(X, g, h, lam)
    if feature is None:
        return node
    left_mask = X[:, feature] <= threshold
    node.is_leaf = False
    node.feature, node.threshold, node.gain = feature, threshold, gain
    node.left = grow_tree(X[left_mask], g[left_mask], h[left_mask], lam, max_depth, depth + 1)
    node.right = grow_tree(X[~left_mask], g[~left_mask], h[~left_mask], lam, max_depth, depth + 1)
    return node

def prune_tree(node, gamma):
    if node.is_leaf:
        return
    prune_tree(node.left, gamma)    # deepest splits resolved first -- bottom-up
    prune_tree(node.right, gamma)
    if node.left.is_leaf and node.right.is_leaf and (node.gain - gamma) < 0:
        node.is_leaf = True         # collapse back; node.value already computed at construction
        node.left = node.right = None

def predict_tree(node, x_row):
    if node.is_leaf:
        return node.value
    branch = node.left if x_row[node.feature] <= node.threshold else node.right
    return predict_tree(branch, x_row)


z0 = np.zeros(4)
g0, h0 = gradient(y_toy, sigmoid(z0)), hessian(sigmoid(z0))
tree = grow_tree(X, g0, h0, lam=1.0, max_depth=2)   # lam=1 here (not 0) to see regularization act
print(f"root gain={tree.gain:.3f} (expect ~1.333); depth-1 gains={tree.left.gain:.3f},{tree.right.gain:.3f} (expect ~-0.267)")

prune_tree(tree, gamma=0.1)
print(f"root kept: {not tree.is_leaf} (gain-gamma={tree.gain-0.1:.3f} >= 0)")
print(f"leaf values after pruning: {tree.left.value:.3f}, {tree.right.value:.3f} (expect -0.667, +0.667)")

# Cover check: root Sigma h = 1.0 (4 pts at p=0.5, h=0.25 each). min_child_weight>0.5 would
# block the depth-1 split -- each resulting child's cover is exactly 0.5
print("cover(root):", np.sum(h0), "| cover(left child):", np.sum(h0[X[:,0]<=5]))